In [1]:
import os
import shutil

import pandas as pd

from project_paths import ensure_pyspark_path, project_root as get_project_root
from pipeline_config import (
    DEFAULT_SPARK_MASTER,
    TEMPERATURE_COLD_MAX,
    TEMPERATURE_GROUP_LABELS,
    TEMPERATURE_MILD_MAX,
    TEMPERATURE_WARM_MAX,
)

ensure_pyspark_path()

base_path = get_project_root()
bronze_path = base_path / 'data' / 'bronze'
silver_path = base_path / 'data' / 'silver'
outputs_tables_path = base_path / 'outputs' / 'tables'

matches_path = bronze_path / 'statsbomb_matches.parquet'
weather_output_path = bronze_path / 'weather_observations.parquet'
weather_features_output_path = silver_path / 'weather_features.parquet'
missing_weather_values_output_path = outputs_tables_path / 'bd12_missing_weather_values.csv'

for path in [silver_path, outputs_tables_path]:
    path.mkdir(parents=True, exist_ok=True)

matches = pd.read_parquet(matches_path)

{
    'weather_input': 'data/bronze/weather_observations.parquet',
    'weather_features_output': 'data/silver/weather_features.parquet',
    'temperature_groups': TEMPERATURE_GROUP_LABELS,
}


{'weather_input': 'data/bronze/weather_observations.parquet',
 'weather_features_output': 'data/silver/weather_features.parquet',
 'temperature_groups': ['cold', 'mild', 'warm', 'hot']}

# 04b Wettervariablen bereinigen und Wettergruppen bilden

Dieser Abschnitt macht die Bronze-Wetterdaten analysefaehig. Die Daten werden aus Parquet gelesen, relevante Spalten werden standardisiert, Gruppen werden aus Temperaturgrenzwerten gebildet und das Ergebnis wird wieder als Parquet geschrieben. Die Grenzwerte sind fachlich explizit: `< 10` ist `cold`, `10 <= x < 20` ist `mild`, `20 <= x <= 28` ist `warm`, und `> 28` ist `hot`.

In [2]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T


def clear_weather_features_output(path: Path) -> None:
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()


SPARK_MASTER = os.getenv('SPARK_MASTER', DEFAULT_SPARK_MASTER)

spark = (
    SparkSession.builder
    .appName('football-weather-bd12-weather-features')
    .master(SPARK_MASTER)
    .getOrCreate()
)
weather_bronze_sdf = spark.read.parquet(str(weather_output_path))

weather_features_sdf = (
    weather_bronze_sdf
    .select(
        F.col('match_id').cast(T.LongType()).alias('match_id'),
        F.to_date('match_date').alias('match_date'),
        F.col('kick_off').cast(T.StringType()).alias('kick_off'),
        F.col('kickoff_timestamp').cast(T.TimestampType()).alias('kickoff_timestamp'),
        F.col('weather_time').cast(T.TimestampType()).alias('weather_time'),
        F.col('hours_from_kickoff').cast(T.DoubleType()).alias('hours_from_kickoff'),
        F.col('weather_lookup_key').cast(T.StringType()).alias('weather_lookup_key'),
        F.col('stadium_id').cast(T.LongType()).alias('stadium_id'),
        F.col('stadium_name').cast(T.StringType()).alias('stadium_name'),
        F.col('city_name').cast(T.StringType()).alias('city_name'),
        F.col('country_name').cast(T.StringType()).alias('country_name'),
        F.col('latitude').cast(T.DoubleType()).alias('latitude'),
        F.col('longitude').cast(T.DoubleType()).alias('longitude'),
        F.col('openmeteo_timezone').cast(T.StringType()).alias('openmeteo_timezone'),
        F.col('temperature_2m').cast(T.DoubleType()).alias('temperature_c'),
        F.col('apparent_temperature').cast(T.DoubleType()).alias('feels_like_c'),
        F.col('precipitation').cast(T.DoubleType()).alias('precipitation_mm'),
        F.coalesce(
            F.col('rain').cast(T.DoubleType()),
            F.col('precipitation').cast(T.DoubleType()),
        ).alias('rain_mm'),
        F.col('request_url').cast(T.StringType()).alias('weather_request_url'),
        F.col('short_name').cast(T.StringType()).alias('tournament_short_name'),
        F.col('competition_name').cast(T.StringType()).alias('competition_name'),
        F.col('season_name').cast(T.StringType()).alias('season_name'),
    )
    .withColumn('rain_flag', F.col('rain_mm') > F.lit(0.0))
    .withColumn(
        'temperature_group',
        F.when(F.col('temperature_c').isNull(), F.lit(None).cast(T.StringType()))
        .when(F.col('temperature_c') < F.lit(TEMPERATURE_COLD_MAX), F.lit('cold'))
        .when(F.col('temperature_c') < F.lit(TEMPERATURE_MILD_MAX), F.lit('mild'))
        .when(F.col('temperature_c') <= F.lit(TEMPERATURE_WARM_MAX), F.lit('warm'))
        .otherwise(F.lit('hot')),
    )
    .withColumn(
        'weather_missing_flag',
        F.col('temperature_c').isNull()
        | F.col('feels_like_c').isNull()
        | F.col('rain_mm').isNull()
        | F.col('rain_flag').isNull()
        | F.col('temperature_group').isNull(),
    )
    .select(
        'match_id',
        'match_date',
        'kick_off',
        'kickoff_timestamp',
        'weather_time',
        'hours_from_kickoff',
        'weather_lookup_key',
        'stadium_id',
        'stadium_name',
        'city_name',
        'country_name',
        'latitude',
        'longitude',
        'openmeteo_timezone',
        'temperature_c',
        'feels_like_c',
        'precipitation_mm',
        'rain_mm',
        'rain_flag',
        'temperature_group',
        'weather_missing_flag',
        'weather_request_url',
        'tournament_short_name',
        'competition_name',
        'season_name',
    )
)

clear_weather_features_output(weather_features_output_path)
weather_features_sdf.write.mode('overwrite').parquet(str(weather_features_output_path))
weather_features = weather_features_sdf.toPandas()

{
    'weather_feature_rows': len(weather_features),
    'weather_features_output_path': 'data/silver/weather_features.parquet',
    'schema': list(weather_features.dtypes.astype(str).items()),
}


{'weather_feature_rows': 314,
 'weather_features_output_path': 'data/silver/weather_features.parquet',
 'schema': [('match_id', 'int64'),
  ('match_date', 'object'),
  ('kick_off', 'object'),
  ('kickoff_timestamp', 'datetime64[ns]'),
  ('weather_time', 'datetime64[ns]'),
  ('hours_from_kickoff', 'float64'),
  ('weather_lookup_key', 'object'),
  ('stadium_id', 'int64'),
  ('stadium_name', 'object'),
  ('city_name', 'object'),
  ('country_name', 'object'),
  ('latitude', 'float64'),
  ('longitude', 'float64'),
  ('openmeteo_timezone', 'object'),
  ('temperature_c', 'float64'),
  ('feels_like_c', 'float64'),
  ('precipitation_mm', 'float64'),
  ('rain_mm', 'float64'),
  ('rain_flag', 'bool'),
  ('temperature_group', 'object'),
  ('weather_missing_flag', 'bool'),
  ('weather_request_url', 'object'),
  ('tournament_short_name', 'object'),
  ('competition_name', 'object'),
  ('season_name', 'object')]}

## Fehlende Wetterwerte dokumentieren und BD-12 pruefen

Fehlende Analysewerte werden als eigene CSV dokumentiert. Auch wenn keine Werte fehlen, wird die Datei mit Header geschrieben, damit der Check reproduzierbar sichtbar bleibt.

In [3]:
missing_value_columns = ['temperature_c', 'feels_like_c', 'rain_mm', 'rain_flag', 'temperature_group']

missing_mask = weather_features[missing_value_columns].isna().any(axis=1)
missing_weather_values = weather_features.loc[
    missing_mask,
    ['match_id', 'match_date', 'stadium_name', 'city_name', 'country_name'],
].copy()
if missing_weather_values.empty:
    missing_weather_values = pd.DataFrame(columns=[
        'match_id', 'match_date', 'stadium_name', 'city_name', 'country_name', 'missing_fields'
    ])
else:
    missing_weather_values['missing_fields'] = [
        ', '.join(column for column in missing_value_columns if pd.isna(row[column]))
        for _, row in weather_features.loc[missing_mask, missing_value_columns].iterrows()
    ]
missing_weather_values.to_csv(missing_weather_values_output_path, index=False)

feature_rows = len(weather_features)
feature_match_count = weather_features['match_id'].nunique()
null_summary = weather_features[missing_value_columns].isna().sum().to_dict()
temperature_group_counts = weather_features['temperature_group'].value_counts(dropna=False).to_dict()

assert feature_rows == matches['match_id'].nunique(), (
    'Weather features should contain one row per match. '
    f"Expected {matches['match_id'].nunique()}, got {feature_rows}."
)
assert feature_match_count == feature_rows, 'Each match should have exactly one weather feature row.'
assert set(weather_features.columns) >= {
    'temperature_c', 'feels_like_c', 'rain_mm', 'rain_flag', 'temperature_group'
}
assert all(value == 0 for value in null_summary.values()), f'Missing BD-12 weather values: {null_summary}'
assert set(temperature_group_counts) <= set(TEMPERATURE_GROUP_LABELS)

{
    'feature_rows': feature_rows,
    'temperature_group_counts': temperature_group_counts,
    'rainy_matches': int(weather_features['rain_flag'].sum()),
    'missing_value_rows': len(missing_weather_values),
    'missing_weather_values_output_path': 'outputs/tables/bd12_missing_weather_values.csv',
}


{'feature_rows': 314,
 'temperature_group_counts': {'warm': 173, 'hot': 91, 'mild': 50},
 'rainy_matches': 40,
 'missing_value_rows': 0,
 'missing_weather_values_output_path': 'outputs/tables/bd12_missing_weather_values.csv'}